<a href="https://colab.research.google.com/github/Rindhya533/CLSA_Assessment_1/blob/main/CLSA_Assessment_Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CLSA ECG Practical Assessment – Part Two -
Name: Morreddigari Rindhya Reddy

Questions

1.   Compute the average trade size (in shares and in AUD) for each venue, both unweighted and weighted by notional traded.
2.   Define short term reversion as the signed relative mid-price movement in basis points (hundredths of a percent) within 5, 10, or 30 seconds after a trade. By definition, the reversion is negative if the mid-price after the trade (i.e. mid at T + 5 seconds, T + 10 seconds, or T + 30 seconds) is worse than the mid-price at the time of the trade (T), and positive if the mid-price after the trade is better than the mid-price at the time of the trade. Compute the average reversion (in basis points) for each venue, both unweighted and weighted by notional traded.
3.	What can you say about the relation exists between relative trade size (trade size divided by median trade size) and reversion for far touch trades on Chi-X Australia?


In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
uploaded = files.upload()

Saving testdata.csv to testdata.csv


In [3]:
df2=pd.read_csv("testdata.csv")

In [4]:
df2.head()

,TRADE_DATE,SIDE,TRADEDQUANTITY,TRADEDPRICE,LAST_MARKET,NEAR_MID_FAR,BID_PRICE_TP00,ASK_PRICE_TP00,MID_PRICE_TP00,MID_PRICE_TP05,MID_PRICE_TP10,MID_PRICE_TP30,MTS
0,2019-09-02,Sell,21.0,3.52,AH,NEAR,3.50,3.52,3.51,3.505,3.505,3.505,NaN
1,2019-09-02,Sell,65.0,3.52,AH,NEAR,3.50,3.52,3.51,3.505,3.505,3.505,NaN
2,2019-09-02,Sell,2451.0,3.52,SEAU,NEAR,3.50,3.52,3.51,3.505,3.505,3.505,NaN
3,2019-09-02,Sell,817.0,3.52,SEAU,NEAR,3.50,3.52,3.51,3.505,3.505,3.505,NaN
4,2019-09-02,Sell,3646.0,3.52,SEAU,FAR,3.52,3.56,3.54,3.505,3.505,3.505,NaN


A new column called "notional" was created by multiplying traded quantity with traded price. This represents the total value traded for each transaction.

In [7]:
df2['notional'] = df2['TRADEDQUANTITY'] * df2['TRADEDPRICE']

Two types of averages are computed:
- Unweighted average
- Weighted average using notional traded value

Trade size is analyzed both in terms of:
- Number of shares traded
- AUD value traded

In [16]:
unweighted_summary = df2.groupby('LAST_MARKET').agg({
    'TRADEDQUANTITY':'mean',
    'notional':'mean'
}).reset_index()
unweighted_summary.columns = [
    'Venue',
    'Avg_Trade_Size_Shares',
    'Avg_Trade_Size_AUD'
]
unweighted_summary

,Venue,Avg_Trade_Size_Shares,Avg_Trade_Size_AUD
0,AH,129.235698,678.578992
1,ASXC,372.427575,647.806989
2,AUCE,285.403497,1177.098467
3,SEAU,272.788134,1551.435308


In [9]:
weighted_summary = df2.groupby('LAST_MARKET').apply(
    lambda x: pd.Series({
        'Weighted_Avg_Shares':
        np.average(
            x['TRADEDQUANTITY'],
            weights=x['notional']
        ),
        'Weighted_Avg_AUD':
        np.average(
            x['notional'],
            weights=x['notional']
        )
    })
).reset_index()
weighted_summary

/tmp/ipykernel_6999/2562869365.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_summary = df2.groupby('LAST_MARKET').apply(


,LAST_MARKET,Weighted_Avg_Shares,Weighted_Avg_AUD
0,AH,1369.772803,4005.548391
1,ASXC,7973.782518,14025.538311
2,AUCE,914.677377,4738.850762
3,SEAU,49244.752079,461260.813219


The direction variable is created to account for Buy and Sell trades separately.
- Buy trades were assigned +1
- Sell/Short Sell trades were assigned -1

In [10]:
df2['direction'] = np.where(
    df2['SIDE'].str.contains('Buy', case=False),1,-1
)

Reversion was calculated as the signed percentage change in mid-price after the trade relative to the mid-price at the trade time.

In [13]:
df2['rev_5s'] = ((df2['MID_PRICE_TP05'] - df2['MID_PRICE_TP00'])/ df2['MID_PRICE_TP00'])*10000*df2['direction']

df2['rev_10s'] = ((df2['MID_PRICE_TP10'] - df2['MID_PRICE_TP00'])/ df2['MID_PRICE_TP00'])*10000*df2['direction']

df2['rev_30s'] = ((df2['MID_PRICE_TP30'] - df2['MID_PRICE_TP00'])/ df2['MID_PRICE_TP00'])*10000*df2['direction']

In [14]:
reversion_summary = df2.groupby('LAST_MARKET').agg({
    'rev_5s': 'mean',
    'rev_10s': 'mean',
    'rev_30s': 'mean'
}).reset_index()
reversion_summary

,LAST_MARKET,rev_5s,rev_10s,rev_30s
0,AH,-1.645505,-1.217832,-1.504501
1,ASXC,-0.615568,-0.889570,-0.926795
2,AUCE,0.335903,0.325375,0.353310
3,SEAU,-0.189374,0.129899,0.463244


Weighted reversion was calculated using notional traded value as weights in order to give larger trades greater influence in the analysis.

In [15]:
weighted_reversion = []
for venue, data in df2.groupby('LAST_MARKET'):
    weighted_rev_5s = np.average(
        data['rev_5s'],
        weights=data['notional']
    )
    weighted_rev_10s = np.average(
        data['rev_10s'],
        weights=data['notional']
    )
    weighted_rev_30s = np.average(
        data['rev_30s'],
        weights=data['notional']
    )
    weighted_reversion.append({
        'Venue': venue,
        'Weighted_Rev_5s': weighted_rev_5s,
        'Weighted_Rev_10s': weighted_rev_10s,
        'Weighted_Rev_30s': weighted_rev_30s
    })
weighted_reversion = pd.DataFrame(weighted_reversion)
weighted_reversion

,Venue,Weighted_Rev_5s,Weighted_Rev_10s,Weighted_Rev_30s
0,AH,-2.555520,-2.563912,-2.778693
1,ASXC,-0.413616,-0.806137,-2.160726
2,AUCE,0.128393,0.111172,0.120284
3,SEAU,1.286628,1.525612,1.729795
